# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates loading and exploring the FAIR^2 dataset with the [`mlcroissant`](https://mlcommons.github.io/croissant/) library. It follows FAIR principles (Findable, Accessible, Interoperable, Reusable) for dataset processing and uses only schema `@id` references, as required for robust reproducibility.

### Dataset Source
The dataset is described by a Croissant schema available at the following URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure the mlcroissant library is installed
!pip install mlcroissant


## 1. Data Loading
Load dataset metadata and overview records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Print core metadata (name and description)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")

## 2. Data Overview
Explore the dataset's record sets, with their associated fields and columns. All entities are referenced by `@id` as required by Croissant specification.

List all available record sets and their field `@id`s.

In [ ]:
# List all record sets in the dataset
record_sets = dataset.record_sets
print(f"Available Record Sets: {len(record_sets)} found.")

for rs in record_sets:
    print(f"---\nRecord set @id: {rs['@id']}")
    print(f"  name: {rs.get('name', '<no name>')}")
    print(f"  description: {rs.get('description', '<no description>')}")
    if 'field' in rs:
        field_ids = rs['field'] if isinstance(rs['field'], list) else [rs['field']]
        print("  Field @ids:")
        for field in field_ids:
            print(f"    - {field}")
    else:
        print("  (No fields listed)")

## 3. Data Extraction
We extract records from the first record set, using only entity `@id`s for all field references as per the specification. You can loop over all record sets if required.

In [ ]:
# Select record set(s) by @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print(f"Using record set @ids: {record_set_ids}")

# Load all record sets into DataFrames
dataframes = {}
for rsid in record_set_ids:
    records = list(dataset.records(record_set=rsid))
    dataframes[rsid] = pd.DataFrame(records)

main_rs = record_set_ids[0]
print(f"Columns for record set {main_rs}:")
print(dataframes[main_rs].columns.tolist())
dataframes[main_rs].head()

## 4. Exploratory Data Analysis (EDA)
Let's look for one or more numeric fields to process. All field access is by column name, which is also the field `@id` according to Croissant loading convention.

You may edit the variable `numeric_field_id` below to point to any numeric field present in your chosen record set. If unsure, review the columns printed above.

In [ ]:
# Pick a record set to analyze (by @id)
rsid = main_rs
df = dataframes[rsid]

# Try to detect numeric columns by dtype (or user can manually edit below)
numeric_fields = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", numeric_fields)
if len(numeric_fields) == 0:
    print("No numeric fields detected. Please check record set or inspect the dataframe, and fill below.")
# For demonstration, pick the first numeric field, else fallback.
numeric_field_id = numeric_fields[0] if numeric_fields else None

if numeric_field_id is not None:
    threshold = df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 10
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records from {rsid} where {numeric_field_id} > {threshold:.3f}:")
    print(filtered_df[[numeric_field_id]].head())
    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Group by a likely categorical field, e.g., 'sample' or 'group', if available
    potential_group_fields = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) or pd.api.types.is_categorical_dtype(df[col])]
    group_field = None
    if potential_group_fields:
        group_field = potential_group_fields[0]
        print(f"Grouping by field: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print("Mean value by group:")
        print(grouped_df.head())
else:
    print("No numeric field to process.")

## 5. Visualization
We plot the distribution of the processed numeric field if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id} in record set {rsid}")
    plt.show()
else:
    print("No numeric field available to plot.")

## 6. Conclusion
Using the `mlcroissant` library, we loaded a FAIR-compliant mass spectrometry dataset, explored its record sets and field `@id`s, and performed basic data extraction and analysis with full schema provenance. Further biological or domain-specific exploration can now proceed on the extracted DataFrames, maintaining linkage to all original data entities using `@id` as specified in the Croissant schema.